In [4]:
import sys
import subprocess
import traceback

# 1. 필수 라이브러리 자동 설치 밑작업
required_libraries = ["webdriver-manager", "selenium", "openpyxl"]
for lib in required_libraries:
    try:
        __import__(lib.replace("-", "_"))
    except ImportError:
        print(f"📦 필수 라이브러리 [{lib}] 설치 중...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", lib])

print("✅ 모든 라이브러리 밑작업 완료!")
print("-" * 50)

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
from openpyxl import Workbook
from datetime import datetime
from webdriver_manager.chrome import ChromeDriverManager

# 사용자 검색어 입력
search_keyword = input("🔍 유튜브에서 검색할 키워드를 입력하세요: ").strip()
if not search_keyword:
    print("❌ 검색어가 입력되지 않아 프로그램을 종료합니다.")
    exit()

options = Options()
options.add_argument("--start-maximized")

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)

wb = Workbook(write_only=True)
ws = wb.create_sheet(title="YouTube Comments")
# 💡 [변경] 엑셀 헤더에 'Likes(좋아요 수)' 컬럼 추가
ws.append(["Video Title", "Video URL", "Likes", "Comment"])

# 유튜브 접속 및 검색
driver.get("https://www.youtube.com")
time.sleep(2)

search_box = driver.find_element(By.NAME, "search_query")
search_box.send_keys(search_keyword)
search_box.send_keys(Keys.RETURN)

# 2. 필터 정렬 안정화 영역
try:
    print("⏳ 필터 버튼 로딩 대기 중...")
    wait = WebDriverWait(driver, 8)
    
    try:
        filter_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//ytd-search-header-renderer//button')))
    except:
        try:
            filter_button = wait.until(EC.element_to_be_clickable((By.XPATH, '//*[@id="filter-button"]//button')))
        except:
            filter_button = wait.until(EC.element_to_be_clickable((By.開, '//button[contains(., "필터") or contains(., "Filters")]'))) if hasattr(By, '開') else wait.until(EC.element_to_be_clickable((By.XPATH, '//button[contains(., "필터") or contains(., "Filters")]')))
    
    driver.execute_script("arguments[0].click();", filter_button)
    print("✅ 필터 버튼 클릭 성공! 정렬 옵션 선택 중...")
    time.sleep(1.5)
    
    try:
        sort_option = wait.until(EC.presence_of_element_located((By.XPATH, '//a[.//yt-formatted-string[text()="인기순"]]')))
    except:
        try:
            sort_option = wait.until(EC.presence_of_element_located((By.XPATH, '//a[.//yt-formatted-string[text()="Relevance"]]')))
        except:
            sort_option = wait.until(EC.presence_of_element_located((By.XPATH, '//ytd-search-filter-options-dialog-renderer//a[contains(., "인기")]')))
            
    driver.execute_script("arguments[0].click();", sort_option)
    print(f"🔥 '{search_keyword}' 인기순 정렬 적용 완료!")
    time.sleep(3)
    
except Exception as e:
    print("❌ 필터 정렬 최종 실패. 기본 검색 결과로 계속 진행합니다.")
    traceback.print_exc()
    print("-" * 50)

# 3. 동영상 링크 수집 (쇼츠 차단 및 일반 영상 10개)
video_links = []
video_titles = []
seen_urls = set()
scroll_attempts = 0

print("🎥 영상 링크 수집 시작 (쇼츠 완벽 차단 모드)...")
while len(video_links) < 50 and scroll_attempts < 50:
    driver.execute_script("window.scrollBy(0, 2500);")
    time.sleep(2)
    
    video_elements = driver.find_elements(By.XPATH, '//ytd-video-renderer')
    
    for video_element in video_elements:
        try:
            title_element = video_element.find_element(By.ID, "video-title")
            url = title_element.get_attribute("href")
            title = title_element.text.strip()
            
            if url and title and "/shorts/" not in url and url not in seen_urls:
                video_links.append(url)
                video_titles.append(title)
                seen_urls.add(url)
                print(f"   👉 수집 성공 ({len(video_links)}/50): {title[:20]}...")
                
            if len(video_links) >= 50:
                break
        except Exception:
            continue
            
    scroll_attempts += 1

print(f"✅ 총 {len(video_links)}개의 영상 링크 수집 완료. 본문 정보 및 댓글 수집을 시작합니다.")

# 4. 각 영상별 좋아요 수 및 댓글 수집 (최대 50개)
for idx in range(len(video_links)):
    print(f"\n[{idx+1}/{len(video_links)}] '{video_titles[idx]}' 분석 중...")
    driver.get(video_links[idx])
    time.sleep(4) # 영상 페이지 로딩 대기
    
    # 💡 [좋아요 수집 핵심 코드] 보내주신 경로 기반 정밀 타겟팅 및 에러 방지
    like_count = "0"
    try:
        # 최근 업데이트된 유튜브의 '좋아요 버튼 컴포넌트'의 aria-label 속성에서 숫자 추출
        like_element = driver.find_element(By.XPATH, '//like-button-view-model//button')
        like_text = like_element.get_attribute("aria-label") # 예: "이 동영상이 마음에 듭니다. 다른 사용자 1,504명과 함께..."
        
        # 텍스트에서 숫자 형태만 정제하거나 글자 그대로 가져오기
        if like_text:
            like_count = like_text.strip()
        else:
            like_count = like_element.text.strip()
            
    except NoSuchElementException:
        # 대체 XPATH 구조 시도
        try:
            like_element = driver.find_element(By.XPATH, '//*[@id="segmented-like-button"]//button')
            like_count = like_element.get_attribute("aria-label") or like_element.text.strip()
        except:
            like_count = "확인 불가(또는 비공개)"
            
    print(f"   👍 좋아요 수집 결과: {like_count}")
    
    # 댓글창 로드를 위한 초기 스크롤
    driver.execute_script("window.scrollTo(0, 500);")
    time.sleep(3)

    comment_set = set()
    prev_count = -1
    stagnant_count = 0

    while len(comment_set) < 50:
        driver.execute_script("window.scrollBy(0, 1000);")
        time.sleep(1.5)
        comments = driver.find_elements(By.CSS_SELECTOR, "#content #content-text")
        for c in comments:
            text = c.text.strip()
            if text and text not in comment_set:
                comment_set.add(text)
                # 💡 [변경] 수집한 좋아요 수(like_count)를 세 번째 칸에 함께 추가하여 저장
                ws.append([video_titles[idx], video_links[idx], like_count, text])
            if len(comment_set) >= 50:
                break
        
        if len(comment_set) == prev_count:
            stagnant_count += 1
        else:
            stagnant_count = 0
            
        if stagnant_count >= 3:
            break
        prev_count = len(comment_set)
    print(f"   💬 댓글 {len(comment_set)}개 수집 완료.")

# 5. 파일 저장
safe_filename = "".join(c for c in search_keyword if c.isalnum() or c in (' ', '_', '-')).strip().replace(' ', '_')
filename = f"{safe_filename}_좋아요포함_댓글50개_영상50개_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx"

wb.save(filename)
driver.quit()
print(f"\n🎉 모든 작업이 완료되었습니다! 엑셀 파일 저장 완료: {filename}")

✅ 모든 라이브러리 밑작업 완료!
--------------------------------------------------
⏳ 필터 버튼 로딩 대기 중...
✅ 필터 버튼 클릭 성공! 정렬 옵션 선택 중...
🔥 '월드컵' 인기순 정렬 적용 완료!
🎥 영상 링크 수집 시작 (쇼츠 완벽 차단 모드)...
   👉 수집 성공 (1/50): 가나, 우루과이 잡으시고, 바지적삼,...
   👉 수집 성공 (2/50): 20년이 지났지만 잊을 수 없는 그 ...
   👉 수집 성공 (3/50): 드라마도 이렇게는 못 쓴다...😂 경...
   👉 수집 성공 (4/50): [MBC 한일월드컵] 2002년 6월...
   👉 수집 성공 (5/50): [2026 FIFA 북중미 월드컵 아...
   👉 수집 성공 (6/50): 명작(名作) 일본만화 월드컵 64강...
   👉 수집 성공 (7/50): [2026 FIFA 북중미 월드컵 아...
   👉 수집 성공 (8/50): 64경기 172골 다 돌려봤습니다.....
   👉 수집 성공 (9/50): 아니 그걸 왜 빼냐고요;; 홍철없는 ...
   👉 수집 성공 (10/50): 지구상 모든 음악들 중 최강자를 가리...
   👉 수집 성공 (11/50): 월클 골키퍼들의 '설명 불가' 반사신...
   👉 수집 성공 (12/50): 이제는 너무 뻔한 클리셰 월드컵...
   👉 수집 성공 (13/50): 최고의 탄수화물 월드컵 (밥 ver....
   👉 수집 성공 (14/50): 어르신은 모르는 MZ 음식 월드컵...
   👉 수집 성공 (15/50): 대한민국 VS 중국 : 2026 FI...
   👉 수집 성공 (16/50): 오천만의 진짜가 되는 시간, FIFA...
   👉 수집 성공 (17/50): 러시아 월드컵 골모음 - 16강 부터...
   👉 수집 성공 (18/50): 심통난 씨네필 셋과 함께하는 명작 영...
   👉 수집 성공 (19/50): 음악 천재들이 한가득했던 2000년대...
   👉 수집 